In [2]:
import pandas as pd
import matplotlib as plt
import json

In [10]:
df_fall_26 = pd.read_csv('/Users/arabellepark/de-densification-2/output/fall_26.csv')

df_spring_26 = pd.read_csv('/Users/arabellepark/de-densification-2/output/spring_26.csv')



In [ ]:
df_fall_26
# df_spring_26


,course_identifier,dept_code,school_code,course_name,term,section,call_number,instructor,start_time,end_time,days
0,AFASG4990,AFAM,SAS,AFR-AMER RESEARCH-WRITING,20263,001,12141,Rachel Grace Newman,12:10,14:00,R 12:10PM-2:00PM
1,AFASG4100,AFAM,SAS,AF-AM: PRO SEMINAR,20263,001,13210,Farah Griffin,14:10,16:00,T 2:10PM-4:00PM
2,AFASG4014,AFAM,SAS,Assurance of Infinity: In the Caribbean,20263,001,14264,Rachel Grace Newman,14:10,16:00,R 2:10PM-4:00PM
3,AFASG4010,AFAM,SAS,Reading the works of Jamaica Kincaid,20263,001,12308,Edwidge Danticat,16:10,18:00,M 4:10PM-6:00PM
4,AFASW4003,AFAM,SAS,Writing and Editing the Black Studies Journal,20263,001,14265,Jafari S Allen,16:10,18:00,T 4:10PM-6:00PM
...,...,...,...,...,...,...,...,...,...,...,...
4413,WMSTW4210,WSTB,BCOL,BLACK GEOGRAPHIES,20263,001,599,Marisa Solomon,16:10,18:00,R 4:10PM-6:00PM
4414,WMSTW3915,WSTB,BCOL,"GENDER, SEXUALITY & POWER IN TRANSNATIONAL PER...",20263,002,897,Manijeh Moradian,12:10,14:00,T 12:10PM-2:00PM
4415,WMSTV3525,WSTB,BCOL,Senior Seminar I (Barnard),20263,001,597,Manijeh Moradian,14:10,16:00,W 2:10PM-4:00PM
4416,WMSTV3525,WSTB,BCOL,Senior Seminar I (Barnard),20263,002,875,Janet Jakobsen,09:00,10:50,W 9:00AM-10:50AM


In [66]:
# grad_pattern = r'.{4}G'
# remove = df['course.course_identifier'].str.contains(grad_pattern, regex=True, na=False)
# df_undergrad = df[~remove]
# df_undergrad

desired_codes = ['SEAS', 'SAS'] 

fall_filtered_df = df_fall_26[df_fall_26['school_code'].isin(desired_codes)]
display(fall_filtered_df.shape[0])
# display(fall_filtered_df)

fall_filtered_df = fall_filtered_df.dropna(subset=['days'])
fall_filtered_df.to_csv('fall_26_school.csv')
display(fall_filtered_df.shape[0])

# spring_filtered_df = df_spring_26[df_spring_26['school_code'].isin(desired_codes)]
# display(spring_filtered_df.shape[0])

# --- 1. LOAD / COPY DATA ---
df = fall_filtered_df.copy()

# --- 2. CLEAN TIMES ---
df['start_time'] = pd.to_datetime(df['start_time'], format='%H:%M')
df['end_time'] = pd.to_datetime(df['end_time'], format='%H:%M')

# drop rows with missing days
df = df.dropna(subset=['days'])

# --- 3. EXPAND DAYS (MW → Monday, Wednesday) ---
day_map = {
    'M': 'Monday',
    'T': 'Tuesday',
    'W': 'Wednesday',
    'R': 'Thursday',
    'F': 'Friday'
}

df['day_list'] = df['days'].apply(
    lambda x: [day_map.get(d) for d in str(x) if d in day_map]
)
df = df.explode('day_list')

# --- 4. CREATE FIXED 30-MIN TIME BINS ---
time_bins = pd.date_range("08:00", "20:00", freq="30min").time

# --- 5. ASSIGN CLASSES TO TIME BINS ---

rows = []

for _, row in df.iterrows():
    day = row['day_list']   # force scalar
    start = row['start_time'].time()
    end = row['end_time'].time()

    for t in time_bins:
        if start <= t < end:
            rows.append({
                'Day': day,
                'Time': t
            })

binned_df = pd.DataFrame(rows)

# --- 6. COUNT CLASSES PER TIME SLOT ---
result = (
    binned_df
    .groupby(['Day', 'Time'])
    .size()
    .reset_index(name='Share')
)

# --- 7. ORDERING FIX ---
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']

result['Day'] = pd.Categorical(result['Day'], categories=day_order, ordered=True)
result['Time'] = pd.to_datetime(result['Time'].astype(str)).dt.time

result = result.sort_values(['Day', 'Time'])

# --- 8. EXPORT ---
result.to_csv('fall_26.csv', index=False)

display(result)

spring_filtered_df = df_spring_26[df_spring_26['school_code'].isin(desired_codes)]
display(spring_filtered_df.shape[0])
# display(fall_filtered_df)

spring_filtered_df = spring_filtered_df.dropna(subset=['days'])
spring_filtered_df.to_csv('spring_26_school.csv')
display(spring_filtered_df.shape[0])


# --- 1. LOAD / COPY DATA ---
df = spring_filtered_df.copy()

# --- 2. CLEAN TIMES ---
df['start_time'] = pd.to_datetime(df['start_time'], format='%H:%M')
df['end_time'] = pd.to_datetime(df['end_time'], format='%H:%M')

# drop rows with missing days
df = df.dropna(subset=['days'])

# --- 3. EXPAND DAYS (MW → Monday, Wednesday) ---
day_map = {
    'M': 'Monday',
    'T': 'Tuesday',
    'W': 'Wednesday',
    'R': 'Thursday',
    'F': 'Friday'
}

df['day_list'] = df['days'].apply(
    lambda x: [day_map.get(d) for d in str(x) if d in day_map]
)
df = df.explode('day_list')

# --- 4. CREATE FIXED 30-MIN TIME BINS ---
time_bins = pd.date_range("08:00", "20:00", freq="30min").time

# --- 5. ASSIGN CLASSES TO TIME BINS ---

rows = []

for _, row in df.iterrows():
    day = row['day_list']   # force scalar
    start = row['start_time'].time()
    end = row['end_time'].time()

    for t in time_bins:
        if start <= t < end:
            rows.append({
                'Day': day,
                'Time': t
            })

binned_df = pd.DataFrame(rows)

# --- 6. COUNT CLASSES PER TIME SLOT ---
result = (
    binned_df
    .groupby(['Day', 'Time'])
    .size()
    .reset_index(name='Share')
)

# --- 7. ORDERING FIX ---
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']

result['Day'] = pd.Categorical(result['Day'], categories=day_order, ordered=True)
result['Time'] = pd.to_datetime(result['Time'].astype(str)).dt.time

result = result.sort_values(['Day', 'Time'])

# --- 8. EXPORT ---
result.to_csv('spring_26.csv', index=False)

display(result)

2630

1911

/var/folders/zl/l01js9_527d_nstkmhybfn0c0000gn/T/ipykernel_76433/1276645140.py:76: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  result['Time'] = pd.to_datetime(result['Time'].astype(str)).dt.time


,Day,Time,Share
25,Monday,08:00:00,15
26,Monday,08:30:00,73
27,Monday,09:00:00,276
28,Monday,09:30:00,315
29,Monday,10:00:00,87
...,...,...,...
20,Friday,18:00:00,7
21,Friday,18:30:00,7
22,Friday,19:00:00,3
23,Friday,19:30:00,3


3700

2227

ValueError: NaTType does not support time

In [ ]:
# from IPython.display import FileLink
# filtered_df.to_csv('SEAS_FAS_only.csv', index=False)
# FileLink('SEAS_FAS_only.csv')

/Users/arabellepark/de-densification-2/SEAS_FAS_only.csv

In [ ]:
prefixes = ('10', '11', '12', '13')
# df['contains'] = df['start_time'].str.startswith(prefixes)
# df

KeyError: 'start_time'

In [17]:
department_summary = df.groupby('department')['contains'].agg(
        total_types='count',
        matching_types='sum'
    )
print(department_summary)

            total_types  matching_types
department                             
AFAM                 12               5
AFSB                 10               6
AHAR                 28              11
AMLP                  6               0
AMSB                 12               6
...                 ...             ...
THEB                 32              11
URSB                 19               7
URSP                 13               6
WRIT                 34               9
WSTB                 17               8

[136 rows x 2 columns]


In [19]:
department_summary['percentage_matching'] = (department_summary['matching_types'] / department_summary['total_types']) * 100

In [21]:
top_20_departments = department_summary.sort_values(by='total_types', ascending=False).head(20)
top_20_departments_2 = department_summary.nlargest(20, 'total_types')

In [23]:
print(top_20_departments)
print(top_20_departments_2)

            total_types  matching_types  percentage_matching
department                                                  
CORE                267             101            37.827715
SOCW                218              31            14.220183
INTA                207              97            46.859903
ARPL                140              71            50.714286
PHYS                130              37            28.461538
FINC                126              14            11.111111
CHEM                111              41            36.936937
EALC                109              62            56.880734
MELC                 93              46            49.462366
ERM                  85               1             1.176471
ECON                 85              31            36.470588
STAT                 80              34            42.500000
COMS                 80              34            42.500000
DROM                 72              16            22.222222
THEA                 69 

In [25]:
df_fall = pd.read_csv('Downloads/all_classes-fall25.csv')
df_fall

,course_identifier,course_name,subject_code,subject_name,school_code,school_name,section,call_number,instructors,enrollment_count,enrollment_max,enrollment_status,days_times,mil_time_from,mil_time_to
0,CHEMW1500,GENERAL CHEMISTRY LABORATORY,CHEM,NaN,SAS,NaN,001,10482,Sarah J Hansen,24.0,46.0,O,T 1:10PM-4:50PM,13:10,16:50
1,CHEMW1500,GENERAL CHEMISTRY LABORATORY,CHEM,NaN,SAS,NaN,002,10483,Joseph C Ulichny,52.0,48.0,F,T 6:10PM-9:50PM,18:10,21:50
2,CHEMW1500,GENERAL CHEMISTRY LABORATORY,CHEM,NaN,SAS,NaN,003,10484,Joseph C Ulichny,51.0,48.0,F,W 1:10PM-4:50PM,13:10,16:50
3,CHEMW1500,GENERAL CHEMISTRY LABORATORY,CHEM,NaN,SAS,NaN,004,10485,Sarah J Hansen,22.0,46.0,O,R 1:10PM-4:50PM,13:10,16:50
4,CHEMW1411,GENERAL CHEMISTRY I - REC,CHEM,NaN,SAS,NaN,001,10521,Robert Beer,20.0,20.0,F,M 4:10PM-5:00PM,16:10,17:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8035,SUSCK5040,SUSTAINABILITY IN THE FACE OF NATURAL DISASTERS,SUSC,NaN,SPEC,NaN,001,12439,"Einat Lev, Chia-Ying Lee",9.0,20.0,O,R 4:10PM-6:00PM,16:10,18:00
8036,SUSCK5010,"CLIMATE SCI FOR DEC MAK: MOD, ANL & AP",SUSC,NaN,SPEC,NaN,001,12336,Yutian Wu,4.0,24.0,O,W 4:10PM-6:00PM,16:10,18:00
8037,SUSCK5999,CAPSTONE WORKSHOP IN SUSTAINABLE SCIENCE,SUSC,NaN,SPEC,NaN,001,12425,"Joseph Paul, Steven N Chillrud",13.0,15.0,O,T 6:10PM-8:00PM,18:10,20:00
8038,SUSCK5995,Internship In SUSC,SUSC,NaN,SPEC,NaN,D01,17079,Karen Acampado,1.0,40.0,O,NaN,NaN,NaN


In [27]:
df_spring = pd.read_csv('Downloads/all_classes-spring26.csv')
df_spring

,course_identifier,course_name,subject_code,subject_name,school_code,school_name,section,call_number,instructors,enrollment_count,enrollment_max,enrollment_status,days_times,mil_time_from,mil_time_to
0,LCRSG8450,PERSPECTVE ON LAT AMER STUDIES,LCRS,NaN,SAS,NaN,001,16449,Gustavo S Azenha,NaN,5.0,O,NaN,NaN,NaN
1,LCRSG6401,LIT/RES-LAT AMER/CARIB STUD II,LCRS,NaN,SAS,NaN,001,16450,Gustavo S Azenha,NaN,5.0,O,W 2:10PM-4:00PM,14:10,16:00
2,LAW,EXTERNSHIP: UNITED NATIONS,LAW,NaN,SLAW,NaN,001,10102,Akshaya Kumar,NaN,999.0,B,NaN,NaN,NaN
3,LAW,EXTERNSHIP: UNITED NATIONS,LAW,NaN,SLAW,NaN,002,10104,Akshaya Kumar,NaN,999.0,B,NaN,NaN,NaN
4,LAW,EXTERNSHIP: UNITED NATIONS,LAW,NaN,SLAW,NaN,003,10103,Jehanne E Henry,NaN,999.0,B,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6322,DVGOU7350,Advanced Economic Development for Internationa...,DVGO,NaN,SIPA,NaN,R02,15587,NaN,NaN,NaN,F,M 3:10PM-5:00PM,15:10,17:00
6323,DVGOU7350,Advanced Economic Development for Internationa...,DVGO,NaN,SIPA,NaN,001,10186,Eric A Verhoogen,29.0,50.0,O,T 1:10PM-3:00PM,13:10,15:00
6324,DVGOU7350,Advanced Economic Development for Internationa...,DVGO,NaN,SIPA,NaN,002,10187,Eric A Verhoogen,31.0,50.0,O,T 3:10PM-5:00PM,15:10,17:00
6325,DVGOU7325,"Inequality, Poverty, and Politics",DVGO,NaN,SIPA,NaN,001,15597,Cesar Zucco,13.0,25.0,O,T 1:10PM-3:00PM,13:10,15:00


In [29]:
course_prefixes = ['MATH', 'POLS', 'ECON', 'PSYC', 'ENGL','HIST']
filtered_df_fall = df_fall[df_fall.course_identifier.str.startswith(tuple(course_prefixes))]
filtered_df_fall['course_identifier']

1214     ECONW3413
1215     ECONW3412
1216     ECONW3412
1217     ECONW3412
1219     ECONG9016
           ...    
7150    HISTBC2695
7151    HISTBC2695
7152    HISTBC2549
7153    HISTBC2413
7154    HISTBC1062
Name: course_identifier, Length: 673, dtype: object

In [31]:
filtered_df_fall.to_csv('filtered_fall_classes.csv', index=False)

In [33]:
course_prefixes = ['MATH', 'POLS', 'ECON', 'PSYC', 'ENGL']
filtered_df_spring = df_spring[df_spring.course_identifier.str.startswith(tuple(course_prefixes))]
filtered_df_spring
filtered_df_spring.to_csv('filtered_spring_classes.csv', index=False)

In [45]:
# analysis of full or over enrolled classes
df_full_spring26 = pd.read_csv('Downloads/spring-26-al.csv')
df_full_spring26

filtered_df_fullspring26 = df_full_spring26[df_full_spring26.enrollment_status.str.startswith(tuple('F'))]
filtered_df_fullspring26

,course_identifier,course_name,subject_code,subject_name,school_code,school_name,section,call_number,instructors,enrollment_count,enrollment_max,enrollment_status,days_times,mil_time_from,mil_time_to
1,AFASG4004,Turn the TV Off: Black Performance Studies,AFAS,NaN,SAS,NaN,001,15974,Johanna F Almiron,17.0,17.0,F,R 12:10PM-2:00PM,12:10,14:00
9,AFASC3011,Spirit of Justice,AFAS,NaN,SAS,NaN,001,17123,Nyle Fort,18.0,18.0,F,M 10:10AM-12:00PM,10:10,12:00
10,AFASW3004,Introduction to Black Geographies,AFAS,NaN,SAS,NaN,001,10496,Brandi Summers,15.0,15.0,F,W 2:10PM-4:00PM,14:10,16:00
12,AFASW1003,Blackness and Frenchness: A Radical Genealogy,AFAS,NaN,SAS,NaN,001,10536,Veronique Charles,18.0,18.0,F,M 12:10PM-2:00PM,12:10,14:00
23,AFASG4080,TOPICS IN THE BLACK EXPERIENCE,AFAS,NaN,SAS,NaN,005,13264,Erica Hunt,1.0,1.0,F,R 4:10PM-6:00PM,16:10,18:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6946,WRITW1100,BEGINNING FICTION WORKSHOP,WRIT,NaN,SART,NaN,005,12807,Haley Garvin,15.0,15.0,F,W 2:10PM-4:00PM,14:10,16:00
6953,WMSTBC3138,AFFECT AND ACTIVISM,WMST,NaN,BCOL,NaN,001,761,Manijeh Moradian,16.0,16.0,F,T 2:10PM-4:00PM,14:10,16:00
6958,WMSTV4905,Labor and Life: Critiques of Capitalism,WMST,NaN,BCOL,NaN,001,767,Neferti Tadiar,16.0,16.0,F,R 12:10PM-2:00PM,12:10,14:00
6959,WMSTW4330,"SWANA Diasporas: Culture, Politics and Identit...",WMST,NaN,BCOL,NaN,001,766,Manijeh Moradian,14.0,14.0,F,R 10:10AM-12:00PM,10:10,12:00


In [51]:
time_counts = filtered_df_fullspring26['mil_time_from'].value_counts().reset_index()
percentages = filtered_df_fullspring26['mil_time_from'].value_counts(normalize=True) * 100
time_counts.columns = ['StartTime', 'Frequency']
percentages.columns = ['StartTime', 'Percentage']
print(time_counts)

   StartTime  Frequency
0      16:10        190
1      10:10        161
2      13:10        124
3      14:10        113
4      12:10         74
5      09:00         73
6      14:40         73
7      18:10         70
8      11:40         63
9      10:00         60
10     08:40         42
11     15:10         39
12     17:10         34
13     14:20         31
14     17:40         30
15     11:00         25
16     14:00         25
17     08:10         23
18     18:20         23
19     18:00         20
20     08:00         16
21     19:10         15
22     13:00         13
23     20:10         11
24     10:50         10
25     11:10          9
26     19:00          7
27     20:30          5
28     19:40          5
29     15:00          4
30     19:30          4
31     17:00          4
32     12:00          4
33     13:20          4
34     09:10          2
35     10:30          2
36     16:00          2
37     09:30          2
38     08:50          2
39     15:45          2
40     18:30    

In [55]:
import datetime as dt
time_buckets = ['8 am - 10 am', '10 am - 12 pm', '12 pm - 2 pm', '2 pm - 4 pm', '4 pm - 6 pm', '6 pm - 8 pm', '8 pm - 10 pm']
filtered_df_fullspring26['start_time'] = pd.to_datetime(filtered_df_fullspring26['mil_time_from'], format='%H:%M').dt.time
bins = [
    dt.time(0, 0),   # 00:00
    dt.time(0, 2),   # 02:00
    dt.time(8, 0),   # 08:00
    dt.time(10, 0),  # 10:00
    dt.time(14, 0),  # 14:00 (2 PM)
    dt.time(16, 0),  # 16:00 (4 PM)
    dt.time(22, 0),  # 22:00 (10 PM)
    dt.time(23, 59, 59) 
]
# creating a column that categorizes the start times as the sole hour value
filtered_df_fullspring26['start_hour'] = pd.to_datetime(filtered_df_fullspring26['mil_time_from'], format='%H:%M').dt.hour
hour_bins = [8, 10, 12, 14, 16, 18, 20, 22]
hour_labels = ['8 am - 10 am', '10 am - 12 pm', '12 pm - 2 pm', '2 pm - 4 pm', '4 pm - 6 pm', '6 pm - 8 pm', '8 pm - 10 pm']
filtered_df_fullspring26['time_bucket'] = pd.cut(filtered_df_fullspring26['start_hour'], bins=hour_bins, labels=hour_labels, right=False, include_lowest=True)

percentages = filtered_df_fullspring26['time_bucket'].value_counts(normalize=True) * 100
print(percentages)

time_bucket
10 am - 12 pm    23.275261
2 pm - 4 pm      20.069686
4 pm - 6 pm      18.257840
12 pm - 2 pm     15.470383
8 am - 10 am     11.428571
6 pm - 8 pm      10.313589
8 pm - 10 pm      1.184669
Name: proportion, dtype: float64


C:\Users\emiik\AppData\Local\Temp\ipykernel_29048\1586038790.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df_fullspring26['start_time'] = pd.to_datetime(filtered_df_fullspring26['mil_time_from'], format='%H:%M').dt.time
C:\Users\emiik\AppData\Local\Temp\ipykernel_29048\1586038790.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df_fullspring26['start_hour'] = pd.to_datetime(filtered_df_fullspring26['mil_time_from'], format='%H:%M').dt.hour
C:\Users\emiik\AppData\Local\Temp\ipyke